# Módulo 06 · Validación, selección y generalización

**Pregunta rectora:** ¿el desempeño observado representa casos futuros o es consecuencia del azar, el sobreajuste, el leakage o una partición incorrecta?

**Criterio de éxito:** construir una estimación fuera de muestra reproducible, sin leakage y vinculada con el costo del error.

**Registro de decisión:** documentar splitter, métrica, baseline, dispersión, umbral operativo y condiciones de revisión.

Este notebook acompaña los 14 laboratorios del módulo y recorre baseline, hold-out, K-Fold, variantes estructurales, pipelines, tuning, Nested CV, OOF, métricas y decisión de negocio.

**Autor:** Sergio Gevatschnaider

In [ ]:
import platform, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import (train_test_split, KFold, LeaveOneOut, StratifiedKFold, GroupKFold, TimeSeriesSplit, RepeatedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_validate, cross_val_predict, learning_curve)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, log_loss, confusion_matrix)
SEED=42
np.random.seed(SEED)
print({'python':platform.python_version(),'sklearn':sklearn.__version__,'seed':SEED})

## 1 · Experimento base
Construimos un problema de regresión reproducible. La meta no es obtener el mejor score posible sino observar cómo cambia la evidencia cuando cambia el protocolo de validación.

In [ ]:
X,y=make_regression(n_samples=900,n_features=12,n_informative=8,noise=28,random_state=SEED)
X=pd.DataFrame(X,columns=[f'x{i:02d}' for i in range(X.shape[1])])
y=pd.Series(y,name='target')
print(X.shape,y.shape)

## 2 · Train, validation y test
El test final tiene valor sólo si permanece fuera de las decisiones de selección. Primero aislamos 20% y trabajamos con el 80% restante.

In [ ]:
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.20,random_state=SEED)
X_train,X_val,y_train,y_val=train_test_split(X_dev,y_dev,test_size=.25,random_state=SEED)
print({'train':len(X_train),'validation':len(X_val),'test':len(X_test)})

## 3 · Baseline antes de optimizar
Una solución sofisticada debe superar una referencia operativa simple. Para regresión, predecir la media es un baseline mínimo.

In [ ]:
baseline=DummyRegressor(strategy='mean').fit(X_train,y_train)
pred_base=baseline.predict(X_val)
print({'MAE':mean_absolute_error(y_val,pred_base),'RMSE':mean_squared_error(y_val,pred_base)**.5,'R2':r2_score(y_val,pred_base)})

## 4 · Hold-out
Un único hold-out es rápido y explicable, pero su estimación depende de qué observaciones quedaron en cada lado.

In [ ]:
lin=LinearRegression().fit(X_train,y_train)
pred_val=lin.predict(X_val)
holdout={'MAE':mean_absolute_error(y_val,pred_val),'RMSE':mean_squared_error(y_val,pred_val)**.5,'R2':r2_score(y_val,pred_val)}
holdout

## 5 · K-Fold Cross-Validation
K-Fold rota el bloque de validación. Cada observación valida una vez y entrena K−1 veces por ciclo.

In [ ]:
kf=KFold(n_splits=5,shuffle=True,random_state=SEED)
cv=cross_validate(LinearRegression(),X_dev,y_dev,cv=kf,scoring={'mae':'neg_mean_absolute_error','r2':'r2'},return_train_score=True)
cv_summary=pd.DataFrame({'fold':range(1,6),'train_r2':cv['train_r2'],'val_r2':cv['test_r2'],'val_mae':-cv['test_mae']})
cv_summary

## 6 · Leer una distribución, no un número
La media resume nivel esperado; el desvío y el rango informan sensibilidad a la partición.

In [ ]:
scores=cv_summary.val_r2.to_numpy()
print(f'R² CV = {scores.mean():.3f} ± {scores.std(ddof=1):.3f}')
print('rango:',scores.min().round(3),'a',scores.max().round(3))
plt.boxplot(scores,vert=False); plt.xlabel('R²'); plt.title('Distribución entre folds'); plt.show()

## 7 · ¿Cuántos folds elegir?
K no es una receta. Cambia el tamaño de validación, el costo y la correlación entre estimaciones.

In [ ]:
rows=[]
for k in [3,5,7,10,15]:
    splitter=KFold(n_splits=k,shuffle=True,random_state=SEED)
    s=cross_val_score(LinearRegression(),X_dev,y_dev,cv=splitter,scoring='r2')
    rows.append({'K':k,'train_%':100*(k-1)/k,'val_%':100/k,'mean_r2':s.mean(),'sd_r2':s.std(ddof=1),'fits':k})
pd.DataFrame(rows)

## 8 · LOOCV
Leave-One-Out es el extremo K=n. Usa casi toda la muestra para entrenar, pero realiza muchos fits y no es automáticamente una mejor decisión.

In [ ]:
X_small=X_dev.iloc[:250]; y_small=y_dev.iloc[:250]
loo=LeaveOneOut()
loo_pred=cross_val_predict(LinearRegression(),X_small,y_small,cv=loo)
print('LOOCV MAE:',round(mean_absolute_error(y_small,loo_pred),3),'| fits:',len(X_small))

## 9 · Clasificación desbalanceada
Creamos un target con 10% de clase positiva para observar por qué la estructura del target importa al partir.

In [ ]:
Xc,yc=make_classification(n_samples=1200,n_features=14,n_informative=7,n_redundant=2,weights=[.90,.10],flip_y=.01,random_state=SEED)
print(pd.Series(yc).value_counts(normalize=True).rename('proportion'))

## 10 · StratifiedKFold
Preserva aproximadamente la prevalencia de clases en cada fold. Esto evita particiones con representación insuficiente de la clase minoritaria.

In [ ]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
props=[]
for i,(_,va) in enumerate(skf.split(Xc,yc),1): props.append({'fold':i,'n':len(va),'positive_rate':yc[va].mean()})
pd.DataFrame(props)

## 11 · GroupKFold y leakage por identidad
Si varias filas pertenecen al mismo cliente, paciente o empresa, compartir la entidad entre train y validation responde una pregunta distinta de generalizar a entidades nuevas.

In [ ]:
Xg,yg=make_classification(n_samples=600,n_features=10,n_informative=6,random_state=SEED)
groups=np.repeat(np.arange(120),5)
gkf=GroupKFold(n_splits=5)
check=[]
for i,(tr,va) in enumerate(gkf.split(Xg,yg,groups),1):
    shared=set(groups[tr]) & set(groups[va])
    check.append({'fold':i,'train_groups':len(set(groups[tr])),'val_groups':len(set(groups[va])),'shared_groups':len(shared)})
pd.DataFrame(check)

## 12 · TimeSeriesSplit
Cuando el tiempo importa, el futuro no puede aparecer en train para evaluar el pasado. El splitter debe reproducir la secuencia real de disponibilidad.

In [ ]:
n=240
t=np.arange(n)
Xts=pd.DataFrame({'trend':t,'season':np.sin(t/12),'lag_proxy':np.cos(t/7)})
yts=0.4*t+15*np.sin(t/12)+np.random.default_rng(SEED).normal(0,5,n)
tscv=TimeSeriesSplit(n_splits=5,gap=3,test_size=30)
[(i+1,tr[0],tr[-1],va[0],va[-1]) for i,(tr,va) in enumerate(tscv.split(Xts))]

## 13 · Leakage por preprocessing global
La regla es simple: todo lo que aprende de los datos debe aprenderse dentro del fold. El ejemplo siguiente muestra el patrón metodológicamente incorrecto.

In [ ]:
scaler=StandardScaler()
X_leaky=scaler.fit_transform(X_dev)  # MAL: fit antes de CV
leaky=cross_val_score(Ridge(alpha=1),X_leaky,y_dev,cv=kf,scoring='r2')
print('Score con preprocessing global:',leaky.mean().round(4))

## 14 · Pipeline: la solución correcta
Pipeline garantiza que fit de imputación, scaling, selección o reducción dimensional ocurra sólo con train de cada fold.

In [ ]:
pipe=Pipeline([('scale',StandardScaler()),('model',Ridge(alpha=1))])
honest=cross_val_score(pipe,X_dev,y_dev,cv=kf,scoring='r2')
print(f'Pipeline CV: {honest.mean():.4f} ± {honest.std(ddof=1):.4f}')

## 15 · Overfitting y complejidad
La señal relevante no es que train sea alto, sino que la brecha train-validation crezca mientras la complejidad aumenta.

In [ ]:
curve=[]
for degree in [1,2,3,5,8]:
    p=Pipeline([('poly',PolynomialFeatures(degree=degree,include_bias=False)),('scale',StandardScaler()),('ridge',Ridge(alpha=10))])
    r=cross_validate(p,X_dev.iloc[:350,:3],y_dev.iloc[:350],cv=5,scoring='r2',return_train_score=True)
    curve.append({'degree':degree,'train':r['train_score'].mean(),'cv':r['test_score'].mean(),'gap':r['train_score'].mean()-r['test_score'].mean()})
pd.DataFrame(curve)

## 16 · Repeated K-Fold
Repetir K-Fold con distintas particiones permite estudiar con mayor resolución cuánto cambia el score por efecto del resampling.

In [ ]:
rkf=RepeatedKFold(n_splits=5,n_repeats=8,random_state=SEED)
rep=cross_val_score(LinearRegression(),X_dev,y_dev,cv=rkf,scoring='r2')
print({'n_scores':len(rep),'mean':rep.mean(),'sd':rep.std(ddof=1),'p05':np.quantile(rep,.05),'p95':np.quantile(rep,.95)})

## 17 · Hyperparameter tuning con GridSearchCV
La búsqueda debe evaluar configuraciones fuera de su propio ajuste. Usamos CV dentro del proceso de selección.

In [ ]:
rf=RandomForestRegressor(n_estimators=120,random_state=SEED,n_jobs=-1)
grid=GridSearchCV(rf,{'max_depth':[3,5,8,None],'min_samples_leaf':[1,3,7]},cv=5,scoring='neg_mean_absolute_error',n_jobs=-1)
grid.fit(X_dev,y_dev)
print(grid.best_params_,'CV MAE:',-grid.best_score_)

## 18 · RandomizedSearchCV
Cuando el espacio es grande, una búsqueda aleatoria puede cubrir más configuraciones con un presupuesto fijo de evaluaciones.

In [ ]:
rnd=RandomizedSearchCV(rf,{'max_depth':[3,4,5,6,8,10,None],'min_samples_leaf':[1,2,3,5,8,12],'max_features':[.5,.7,1.0]},n_iter=10,cv=5,scoring='neg_mean_absolute_error',random_state=SEED,n_jobs=-1)
rnd.fit(X_dev,y_dev)
print(rnd.best_params_,'CV MAE:',-rnd.best_score_)

## 19 · Nested Cross-Validation
Si usamos el mismo CV para probar muchas configuraciones y para reportar la ganadora, introducimos optimismo por selección. Nested CV separa ambos roles.

In [ ]:
inner=KFold(n_splits=3,shuffle=True,random_state=SEED)
outer=KFold(n_splits=5,shuffle=True,random_state=2026)
search=GridSearchCV(RandomForestRegressor(n_estimators=100,random_state=SEED,n_jobs=-1),{'max_depth':[3,6,None],'min_samples_leaf':[1,5]},cv=inner,scoring='neg_mean_absolute_error',n_jobs=-1)
nested=cross_val_score(search,X_dev,y_dev,cv=outer,scoring='neg_mean_absolute_error',n_jobs=-1)
print(f'Nested CV MAE = {-nested.mean():.2f} ± {nested.std(ddof=1):.2f}')

## 20 · Predicciones OOF
Cada observación recibe una predicción de un modelo que no la utilizó para entrenar. Esto permite diagnósticos globales sin mezclar predicciones in-sample.

In [ ]:
oof=cross_val_predict(pipe,X_dev,y_dev,cv=kf,n_jobs=-1)
oof_df=pd.DataFrame({'y':y_dev.to_numpy(),'pred_oof':oof})
oof_df['error']=oof_df.y-oof_df.pred_oof
oof_df.head()

## 21 · Métricas de clasificación
La mejor métrica depende del error que cuesta. Accuracy puede ocultar fallas graves en clases minoritarias.

In [ ]:
clf=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=2000))])
prob=cross_val_predict(clf,Xc,yc,cv=skf,method='predict_proba',n_jobs=-1)[:,1]
pred=(prob>=.5).astype(int)
print({'accuracy':accuracy_score(yc,pred),'precision':precision_score(yc,pred),'recall':recall_score(yc,pred),'f1':f1_score(yc,pred),'roc_auc':roc_auc_score(yc,prob),'pr_auc':average_precision_score(yc,prob),'log_loss':log_loss(yc,prob)})

## 22 · Métricas de regresión
MAE expresa error absoluto medio; RMSE penaliza más los errores grandes; R² compara contra una referencia basada en la media. Ninguna reemplaza el contexto de negocio.

In [ ]:
print({'OOF_MAE':mean_absolute_error(y_dev,oof),'OOF_RMSE':mean_squared_error(y_dev,oof)**.5,'OOF_R2':r2_score(y_dev,oof)})

## 23 · Threshold como decisión de negocio
Una probabilidad no es una acción. El umbral debe elegirse según costos, capacidad operativa y prevalencia, no por la costumbre de usar 0.5.

In [ ]:
rows=[]
for th in np.arange(.10,.91,.10):
    pr=(prob>=th).astype(int); tn,fp,fn,tp=confusion_matrix(yc,pr).ravel()
    cost=fp*1+fn*8
    rows.append({'threshold':round(th,2),'precision':precision_score(yc,pr,zero_division=0),'recall':recall_score(yc,pr),'FP':fp,'FN':fn,'cost':cost})
pd.DataFrame(rows).sort_values('cost').head()

## 24 · Estabilidad por segmentos
Una media global puede esconder poblaciones donde el modelo falla. Con OOF podemos revisar error por segmentos sin usar predicciones del propio entrenamiento.

In [ ]:
rng=np.random.default_rng(SEED)
oof_df['segment']=rng.choice(['A','B','C'],size=len(oof_df),p=[.5,.3,.2])
segment_report=oof_df.groupby('segment').apply(lambda d:pd.Series({'n':len(d),'MAE':mean_absolute_error(d.y,d.pred_oof),'bias':(d.pred_oof-d.y).mean()}),include_groups=False)
segment_report

## 25 · Learning curve
Si train y validation convergen en un nivel pobre, puede faltar capacidad; si hay una brecha grande, puede faltar regularización o datos. La curva evita diagnosticar sólo con intuición.

In [ ]:
sizes,tr,va=learning_curve(pipe,X_dev,y_dev,cv=5,scoring='r2',train_sizes=np.linspace(.2,1,5),n_jobs=-1)
plt.plot(sizes,tr.mean(1),marker='o',label='train'); plt.plot(sizes,va.mean(1),marker='o',label='validation'); plt.xlabel('n entrenamiento'); plt.ylabel('R²'); plt.legend(); plt.show()

## 26 · Checklist metodológico
Antes de confiar en un score, el protocolo debe declarar unidad de generalización, disponibilidad de información, splitter, preprocessing, baseline, métrica, dispersión y capa final de evaluación.

In [ ]:
checklist=['Unidad de observación y de generalización definidas','Test final reservado cuando corresponde','Splitter reproduce entidad/tiempo real','Preprocessing dentro del Pipeline','Baseline explícito','Media + dispersión reportadas','Métrica ligada al costo del error','Tuning separado de evaluación','Semillas y versiones registradas','Plan de monitoreo de drift']
for i,x in enumerate(checklist,1): print(f'{i:02d}. {x}')

## 27 · Business Lab
La decisión final combina desempeño, estabilidad y umbral operativo. Un buen score técnico no implica automáticamente que el modelo esté listo para producción.

In [ ]:
decision=pd.DataFrame([{'modelo':'Baseline','score':0.61,'stability':0.96,'operational_fit':0.90},{'modelo':'Modelo A','score':0.82,'stability':0.91,'operational_fit':0.88},{'modelo':'Modelo B','score':0.85,'stability':0.69,'operational_fit':0.72}])
decision['business_index']=.5*decision.score+.3*decision.stability+.2*decision.operational_fit
decision.sort_values('business_index',ascending=False)

## 28 · Test final y cierre
Sólo después de cerrar selección, features, preprocessing y criterios de decisión se toca el test reservado. Ese resultado responde una pregunta distinta: ¿cómo rindió el procedimiento elegido sobre datos completamente fuera de la selección?

**Idea final:** validar no es un trámite técnico; es diseñar evidencia que se parezca al futuro donde el modelo deberá sostener decisiones.

In [ ]:
final_model=grid.best_estimator_.fit(X_dev,y_dev)
final_pred=final_model.predict(X_test)
final_report={'test_MAE':mean_absolute_error(y_test,final_pred),'test_RMSE':mean_squared_error(y_test,final_pred)**.5,'test_R2':r2_score(y_test,final_pred)}
print(final_report)
print('\nCierre: compare test final con CV, documente diferencias y defina condiciones de monitoreo/reentrenamiento.')